# Практика MT

В этом ноутбуке -- практические задания для закрепления материала про машинный перевод. Попробуйте сначала выполнить задачи сами, используя документацию и формулы из теории, или сразу разбирайте приведённые примеры кода.

### 1. Попробуем реализовать seq2seq модель с attention на PyTorch

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import random

# Параметры модели
INPUT_DIM = 10      # размер словаря входной последовательности
OUTPUT_DIM = 10     # размер словаря выходной последовательности
HID_DIM = 16        # размер скрытого состояния GRU
SEQ_LEN = 5         # длина входной последовательности для примера
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class Encoder(nn.Module):
    def __init__(self, input_dim, hid_dim):
        super().__init__()
        # Слой для преобразования индексов слов в векторы фиксированной размерности
        self.embedding = nn.Embedding(input_dim, hid_dim)
        # GRU для обработки последовательности
        self.rnn = nn.GRU(hid_dim, hid_dim)

    def forward(self, src):
        # YOUR CODE HERE
        return outputs, hidden

class Attention(nn.Module):
    def __init__(self, hid_dim):
        super().__init__()
        # Линейный слой для вычисления "энергии" внимания
        self.attn = nn.Linear(hid_dim*2, hid_dim)
        # Линейный слой для преобразования энергии в одно число (вес внимания)
        self.v = nn.Linear(hid_dim,1,bias=False)

    def forward(self, hidden, encoder_outputs):
        # YOUR CODE HERE

class Decoder(nn.Module):
    def __init__(self, output_dim, hid_dim, attention):
        super().__init__()
        self.output_dim = output_dim
        self.attention = attention
        self.embedding = nn.Embedding(output_dim, hid_dim)  # эмбеддинг для целевой последовательности
        self.rnn = nn.GRU(hid_dim*2, hid_dim)  # GRU получает вектор + контекстное внимание
        self.fc_out = nn.Linear(hid_dim*2, output_dim)  # линейный слой для предсказания следующего слова

    def forward(self, input, hidden, encoder_outputs):
        # YOUR CODE HERE

In [ ]:
# @title
import torch
import torch.nn as nn
import torch.optim as optim
import random

# Параметры модели
INPUT_DIM = 10      # размер словаря входной последовательности
OUTPUT_DIM = 10     # размер словаря выходной последовательности
HID_DIM = 16        # размер скрытого состояния GRU
SEQ_LEN = 5         # длина входной последовательности для примера
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class Encoder(nn.Module):
    def __init__(self, input_dim, hid_dim):
        super().__init__()
        # Слой для преобразования индексов слов в векторы фиксированной размерности
        self.embedding = nn.Embedding(input_dim, hid_dim)
        # GRU для обработки последовательности
        self.rnn = nn.GRU(hid_dim, hid_dim)

    def forward(self, src):
        # src: [seq_len, batch_size]
        embedded = self.embedding(src)  # Преобразуем индексы в эмбеддинги
        outputs, hidden = self.rnn(embedded)  # Пропускаем через GRU
        # outputs: [seq_len, batch_size, hid_dim] — все скрытые состояния
        # hidden: [1, batch_size, hid_dim] — последнее скрытое состояние
        return outputs, hidden

class Attention(nn.Module):
    def __init__(self, hid_dim):
        super().__init__()
        # Линейный слой для вычисления "энергии" внимания
        self.attn = nn.Linear(hid_dim*2, hid_dim)
        # Линейный слой для преобразования энергии в одно число (вес внимания)
        self.v = nn.Linear(hid_dim,1,bias=False)

    def forward(self, hidden, encoder_outputs):
        # hidden: [1, batch_size, hid_dim] — текущее скрытое состояние декодера
        # encoder_outputs: [seq_len, batch_size, hid_dim] — все скрытые состояния энкодера
        src_len = encoder_outputs.shape[0]

        # Повторяем скрытое состояние для каждого временного шага входа
        hidden = hidden.repeat(src_len, 1, 1)

        # Вычисляем энергию внимания
        energy = torch.tanh(self.attn(torch.cat((hidden, encoder_outputs), dim=2)))
        # Преобразуем энергию в веса внимания с помощью softmax
        attention = torch.softmax(self.v(energy).squeeze(2), dim=0)
        # attention: [seq_len, batch_size] — веса внимания для каждого шага
        return attention

class Decoder(nn.Module):
    def __init__(self, output_dim, hid_dim, attention):
        super().__init__()
        self.output_dim = output_dim
        self.attention = attention
        self.embedding = nn.Embedding(output_dim, hid_dim)  # эмбеддинг для целевой последовательности
        self.rnn = nn.GRU(hid_dim*2, hid_dim)  # GRU получает вектор + контекстное внимание
        self.fc_out = nn.Linear(hid_dim*2, output_dim)  # линейный слой для предсказания следующего слова

    def forward(self, input, hidden, encoder_outputs):
        # input: [batch_size] — текущий токен на входе
        input = input.unsqueeze(0)  # преобразуем в форму [1, batch_size]
        embedded = self.embedding(input)  # [1, batch_size, hid_dim]

        # Вычисляем внимание
        a = self.attention(hidden, encoder_outputs)  # [seq_len, batch_size]
        a = a.unsqueeze(1).permute(1,2,0)  # [batch_size, 1, seq_len]

        encoder_outputs = encoder_outputs.permute(1,2,0)  # [batch_size, hid_dim, seq_len]
        weighted = torch.bmm(a, encoder_outputs).permute(2,0,1)  # контекстный вектор [1, batch, hid_dim]

        # Объединяем эмбеддинг текущего токена и контекст внимания
        rnn_input = torch.cat((embedded, weighted), dim=2)

        # Пропускаем через GRU
        output, hidden = self.rnn(rnn_input, hidden)

        # Предсказываем следующий токен
        prediction = self.fc_out(torch.cat((output, weighted), dim=2).squeeze(0))  # [batch, output_dim]
        return prediction, hidden

# Пример использования
encoder = Encoder(INPUT_DIM, HID_DIM).to(device)
attn = Attention(HID_DIM).to(device)
decoder = Decoder(OUTPUT_DIM, HID_DIM, attn).to(device)

# Создаем случайный вход (например, последовательность индексов)
src = torch.randint(0, INPUT_DIM, (SEQ_LEN,1)).to(device)

# Пропускаем через энкодер
encoder_outputs, hidden = encoder(src)

# Начальный токен для декодера (обычно <sos>)
dec_input = torch.tensor([0]).to(device)

# Получаем предсказание от декодера
output, hidden = decoder(dec_input, hidden, encoder_outputs)

print("Output shape:", output.shape)  # [batch_size, OUTPUT_DIM]

### 2. Используем pretrained модель MarianMT для практического перевода

In [ ]:
from transformers import MarianMTModel, MarianTokenizer


model_name = "Helsinki-NLP/opus-mt-en-ru"
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name).to(device)


def translate(texts):
 #YOUR CODE HERE

In [ ]:
# @title
from transformers import MarianMTModel, MarianTokenizer


model_name = "Helsinki-NLP/opus-mt-en-ru"
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name).to(device)


def translate(texts):
tokens = tokenizer(texts, return_tensors="pt", padding=True).to(device)
translated_tokens = model.generate(**tokens)
return [tokenizer.decode(t, skip_special_tokens=True) for t in translated_tokens]


texts = ["Hello world!", "How are you?"]
translations = translate(texts)
for t, tr in zip(texts, translations):
print(f"{t} -> {tr}")

### 3. Расчёт Cross-Entropy Loss на простом примере

Cross-Entropy Loss - стандартная функция потерь для задач классификации, включая распознавание речи и машинный перевод. Она измеряет расстояние между истинным распределением классов и предсказанным моделью распределением.

Если модель уверена и предсказывает правильный класс -> loss маленький.

Если модель ошибается -> loss большой.

In [ ]:
logits = torch.tensor([[2.0, 0.5, 0.1]]) # пример предсказанных логитов
target = torch.tensor([0]) # правильный токен

# YOUR CODE HERE

In [ ]:
# @title
logits = torch.tensor([[2.0, 0.5, 0.1]]) # пример предсказанных логитов
target = torch.tensor([0]) # правильный токен

loss_fn = nn.CrossEntropyLoss()
loss = loss_fn(logits, target)
print("Cross-Entropy Loss:", loss.item())

### 4. Оценка качества перевода с помощью SacréBLEU

In [ ]:
references = [
"Это пример машинного перевода.",
"Машинный перевод является сложной задачей.",
"NLP развивается очень быстро."
]


hypotheses = [
"Это пример перевода машиной.",
"Машинный перевод сложная задача.",
"Естественная обработка языка развивается быстро."
]


list_references = [[ref] for ref in references]


# YOUR CODE HERE

In [ ]:
# @title
references = [
"Это пример машинного перевода.",
"Машинный перевод является сложной задачей.",
"NLP развивается очень быстро."
]


hypotheses = [
"Это пример перевода машиной.",
"Машинный перевод сложная задача.",
"Естественная обработка языка развивается быстро."
]

# SacréBLEU требует список списков для reference
list_references = [[ref] for ref in references]

# Вычисляем метрику BLEU (Bilingual Evaluation Understudy)
# BLEU сравнивает n-граммы (обычно до 4-словных) гипотез с эталоном
# Чем выше BLEU, тем ближе перевод к эталону (максимум 100)
bleu = sacrebleu.corpus_bleu(hypotheses, list_references)
print(f"BLEU: {bleu.score:.2f}")

# Вычисляем CHRF (Character n-gram F-score)
# CHRF — метрика, основанная на совпадении n-грамм символов
# Хорошо подходит для языков с богатой морфологией (русский)
chrf = sacrebleu.corpus_chrf(hypotheses, list_references)
print(f"CHRF: {chrf.score:.2f}")

# Вычисляем TER (Translation Edit Rate)
# TER показывает, сколько правок нужно внести, чтобы перевод совпал с эталоном
# Чем ниже TER, тем лучше (0% — идеальный перевод)
ter = sacrebleu.corpus_ter(hypotheses, list_references)
print(f"TER: {ter.score:.2f}")